# 01 - Data and splits

Downloads the CFPB complaints, removes noisy rows, builds the category list (taxonomy) and splits the
data into train / tuning / test. **Cost: $0.** Takes about 5-15 minutes. Safe to rerun: finished steps
are skipped.

**Step 1 - setup (free).** Downloads the project code from GitHub, installs it, connects Google Drive
(all results are saved in `MyDrive/cfpb-nlp-classifier/phase1`), reads your `GEMINI_API_KEY` secret (and the optional `TYPESAFE_API_KEY` Jev secret) and
logs in to Google Sheets. Colab will ask you to allow Drive and Sheets access - click Allow.
Takes 2-4 minutes the first time. If it says the key is missing: click the key icon on the left of
Colab, add a secret named `GEMINI_API_KEY`, and switch "Notebook access" on.
On the first run after a project update, setup may say it moved old results into an `archive_...`
folder on Drive. That is expected and needs nothing from you: your spending record, saved Gemini
answers and saved choices stay where they are.

In [ ]:
!git clone -q https://github.com/Dentoneer/cfpb-nlp-classifier /content/cfpb-nlp-classifier || git -C /content/cfpb-nlp-classifier pull -q
# Installs this project only; never upgrades Colab's own Google libraries.
%pip install -q -e "/content/cfpb-nlp-classifier[ml]"
import sys; sys.path.insert(0, "/content/cfpb-nlp-classifier")
from nlp.colab import setup
cfg, gc, api_key, jev_key = setup()

**Choose the date window (free).** Looks at how many complaints exist per month and picks the latest
stretch of up to 12 full months where the product/issue list is stable. **Look at:** the printed window.
If it starts after 2024-01-01 it is used instead of the default start.

In [ ]:
import dataclasses
from datetime import date
from nlp import colab, store, pipeline as P
from nlp.data import HF_SOURCE, FilterParams, max_date_received, product_counts_by_month
from nlp.taxonomy import choose_window
params = FilterParams()
counts = product_counts_by_month(HF_SOURCE, params)
start, end = choose_window(counts, max_date_received(HF_SOURCE))
print("Stable window:", start, "to", end, "(end excluded)")
if start > date.fromisoformat(params.start):
    params = dataclasses.replace(params, start=start.isoformat())
print("Using FilterParams.start =", params.start)

**Load and clean the complaints (free, slowest step).** Applies the noise filters (empty text, copy-pasted
templates, very short or mostly redacted text, complaints without a sub-issue (`no_sub_issue`), rare
categories). **Look at:** the table - each row
shows how many complaints that filter removed, and the last row should leave several thousand complaints.

In [ ]:
complaints, report = colab.load_complaints(cfg, HF_SOURCE, params)
print(len(complaints), "complaints kept")
report

**Build the taxonomy and publish it to Google Sheets (free).** The taxonomy is the list of every
Product / Issue / Sub-issue combination. Each row also has a `label` (`Product|Issue|Sub-issue`) - the
exact text Gemini answers with. **Look at:** the printed Sheet link - open it to see the list. Notebook 03
adds a description to every row of this same Sheet.

In [ ]:
from nlp.taxonomy import build_taxonomy
from nlp.sheets import publish_taxonomy
taxonomy = store.stage(cfg, P.STAGE_TAXONOMY, lambda: build_taxonomy(complaints))
print(len(taxonomy), "category paths")
print("Taxonomy sheet:", colab.publish_once(cfg, "taxonomy", lambda: publish_taxonomy(
    gc, taxonomy, f"CFPB taxonomy ({cfg.data_version})")))

**Make the splits (free).** Randomly (with a fixed seed) sets aside about 200 tuning complaints, 800 test
complaints (never used for tuning) and up to 20,000 for training the classic models.
**Look at:** the table - every Product should appear in all three columns.

In [ ]:
from nlp.sampling import make_splits
splits = store.stage(cfg, P.STAGE_SPLITS, lambda: make_splits(complaints, cfg))
colab.split_summary(complaints, splits)

In [ ]:
from nlp import colab
ledger, *_ = colab.open_tools(cfg)
print(colab.ledger_report(ledger))

Done. Next: `02_baselines.ipynb` (switch the runtime to a T4 GPU first).